This notebook introduces `model.find()` and `model.get()` for navigating model elements by qualified name; after running it you can inspect any named element in the cumulative model without knowing its position in the query result list.

Chapter 5 shifts from defining the model to inspecting and extending it. The cumulative model has fifteen named elements spanning five kinds: part definitions, part usages, a requirement, a calc def, an action def, and item definitions. Navigating by position is fragile; navigating by qualified name (`package::element`) is stable across additions.

`model.find(name)` returns a `Symbol` or `None` for a short or fully-qualified name. `model.get(fqn)` returns a `Symbol` and raises if the name is absent. Together they provide the navigation layer Ch5 and Ch6 build on.

In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """
package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
    action def ApplyHeat {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        out energy : Real;
        first start;
        then action calculate {
            assign energy := DeliveredEnergy(power, duration, efficiency);
        }
        then done;
    }
    item def Start;
    item def Finish;
    item def Cancel;
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"
print(f"Model ok: {model.ok}")

In [ ]:
# Negative control: specializing from an undefined type raises "unresolved reference".
# find/get can only navigate elements that parsed successfully — this confirms
# the model must be valid before navigation is meaningful.
bad_source = """
package BadNav {
    private import ScalarValues::*;
    part def Probe :> UndefinedBase;
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print(f"Neg control diagnostics: {bad.diagnostics[0].message!r}")

In [ ]:
# Navigate to the Heater part definition
heater = model.find("ToasterDemo::Heater")
print(f"find result: id={heater.id!r}, kind={heater.kind!r}")

# Navigate to the DeliveredEnergy calc def by FQN
energy_calc = model.get("ToasterDemo::DeliveredEnergy")
print(f"get result: id={energy_calc.id!r}, kind={energy_calc.kind!r}")

# model.find returns None for unknown names (no exception)
missing = model.find("ToasterDemo::Nonexistent")
assert missing is None
print(f"missing element: {missing}")

The qualified-name addressing scheme in SysML v2 (A-F) is traversed by `model.find()` and `model.get()` in OpenSysML (O-S); the symbol's `id` and `kind` are printed, confirming the element is present and correctly typed (E).

Try the chapter exercise in `exercises/ch05/exercise.ipynb`: use `model.find()` to navigate to `CoffeeDemo::CoffeeFlow` after building the assembly, then confirm `model.find()` returns `None` for an element that does not exist.